In [45]:
import os
import json 
import getpass
import logging
from pathlib import Path
from time import sleep
import subprocess
import sys

from langchain_core.tools import Tool
from langgraph.graph import StateGraph, END
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain.callbacks import get_openai_callback
from langchain.tools import BaseTool

from typing import TypedDict, Dict, List

from PyDI.io import load_xml, load_parquet, load_csv
from PyDI.profiling import DataProfiler
from blocking_tester import BlockingTester
from matching_tester import MatchingTester
from PyDI.entitymatching import RuleBasedMatcher
from schema_matching_node import run_schema_matching

## Initialize

In [46]:
OUTPUT_DIR = "output/"
INPUT_DIR = "input/"

INCLUDE_DOCS = False # IMPORTANT: Use carefully since token usage is increased drastically with documentation
USE_LLM = "gpt"
#USE_LLM = "gemini"
#USE_LLM = "groq"

In [47]:
if USE_LLM == "gemini": # or USE_LLM == "gemini_broken":
    if "GOOGLE_API_KEY" not in os.environ:
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")
elif USE_LLM == "groq":
    if "GROQ_API_KEY" not in os.environ:
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")
elif USE_LLM == "gpt":
    if "OPENAI_API_KEY" not in os.environ:
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

In [48]:
logging.basicConfig(filename= OUTPUT_DIR + 'agent.log',
                    filemode='a',
                    format='%(asctime)s,%(msecs)d %(name)s %(levelname)s %(message)s',
                    datefmt='%H:%M:%S',
                    level=logging.DEBUG,
                    encoding='utf-8')

## Utilities

In [49]:
def load_dataset(path):
    # check file exists
    if not os.path.exists(path):
        raise FileNotFoundError(f"Dataset not found at: {path}")
    ext = os.path.splitext(path)[1].lower()

    # load dataset according to extension
    if ext == ".parquet":
        df = load_parquet(path)
    elif ext == ".csv":
        df = load_csv(path)
    elif ext == ".xml":
        df = load_xml(path, nested_handling="aggregate")
    else:
        raise ValueError(f"Unsupported format: {ext}. Supported: .csv, .parquet, .xml")
    return df
    

## Tools

In [50]:
class ProfileDatasetTool(BaseTool):
    name: str = "profile_dataset"
    description: str = """
        A tool that takes the path as a argument called `path` of type str of the dataset file as string and performs data analysis. 
        A JSON string is returned with the profile data.
    """
    
    def _run(self, path) -> str:   
        return self.create_profile(path)

    def create_profile(self, path):

        if not path or not isinstance(path, str):
                raise ValueError("ProfileDatasetTool requires a single path string argument.")

        df = load_dataset(path)    
        
        if df is None or getattr(df, "empty", False):
            # return a structured JSON error string (LLM will see this as content)
            return json.dumps({"error": f"Dataset at {path} loaded as empty or failed to load."})

        profiler = DataProfiler()
        profile = profiler.summary(df, print_summary=False)

        # ensure to return a JSON string (your docstring promised JSON string)
        try:
            profile_json = json.dumps(profile, default=str)
        except Exception:
            # fallback: convert to str if not json-serializable
            profile_json = json.dumps({"profile_str": str(profile)})

        return profile_json
        

## Agents

In [91]:
class SimpleModelAgentState(TypedDict):
    datasets: list
    entity_matching_testsets: list
    fusion_testset: str
    blocking_config: Dict  # NEW: Blocking config from BlockingTester
    matching_config: Dict  # NEW: Matching config from MatchingTester
    matcher_mode: str  # NEW: Matcher mode (rule_based/ml/auto)
    schema_correspondences: Dict  # NEW: Schema matching results
    
    data_profiles: Dict
    
    integration_pipeline_code: str
    pipeline_execution_result: str
    pipeline_execution_attempts: int

    evaluation_code: str
    evaluation_execution_result: str
    evaluation_execution_attempts: int

    evaluation_attempts: int
    evaluation_metrics: Dict
    evaluation_analysis: str

class SimpleModelAgent:
    
    def __init__(self, model, profileTool):
        # initialize logger
        self.logger = logging.getLogger()
        self.base_model = model
        self.token_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0, "total_cost": 0.0}
        
        # prepare the StateGraph
        graph = StateGraph(SimpleModelAgentState)

        # create nodes
        graph.add_node("match_schemas", self.match_schemas)
        graph.add_node("profile_data", self.profile_data)
        graph.add_node("run_blocking_tester", self.run_blocking_tester)
        graph.add_node("run_matching_tester", self.run_matching_tester)
        graph.add_node("pipeline_adaption", self.pipeline_adaption)
        graph.add_node("execute_pipeline", self.execute_pipeline)
        graph.add_node("evaluation_adaption", self.evaluation_adaption)
        graph.add_node("execute_evaluation", self.execute_evaluation)
        graph.add_node("evaluation_decision", self.evaluation_decision)
        graph.add_node("evaluation_reasoning", self.evaluation_reasoning)

        # create edges
        graph.add_edge("match_schemas", "profile_data")
        graph.add_edge("profile_data", "run_blocking_tester")
        graph.add_edge("run_blocking_tester", "run_matching_tester")
        graph.add_edge("run_matching_tester", "pipeline_adaption")
        graph.add_edge("pipeline_adaption", "execute_pipeline")
        
        graph.add_conditional_edges(
            "execute_pipeline",
            lambda state: (
                "evaluation_adaption"
                if isinstance(state.get("pipeline_execution_result", ""), str)
                   and state["pipeline_execution_result"].lower().startswith("success")
                else "pipeline_adaption"
                if state.get("pipeline_execution_attempts", 0) < 5
                else END
            ),
            {
                "pipeline_adaption": "pipeline_adaption",
                "evaluation_adaption": "evaluation_adaption",
                END: END
            }
        )

        graph.add_edge("evaluation_adaption", "execute_evaluation")

        graph.add_conditional_edges(
            "execute_evaluation",
            lambda state: (
                "evaluation_decision" 
                if isinstance(state.get("evaluation_execution_result", ""), str)
                   and state["evaluation_execution_result"].lower().startswith("success")
                else "evaluation_adaption"
                if state.get("evaluation_attempts", 0) < 3
                else END
            ),
            {
                "evaluation_adaption": "evaluation_adaption",
                "evaluation_decision": "evaluation_decision",
                END: END
            }
        )


        graph.add_conditional_edges(
            "evaluation_decision",
            lambda state: (
                "evaluation_reasoning"
                if state.get("evaluation_metrics", {}).get("overall_accuracy", 0) < 0.85
                   and state.get("evaluation_attempts", 0) < 3
                else END
            ),
            {
                "evaluation_reasoning": "evaluation_reasoning",
                END: END
            }
        )

        graph.add_edge("evaluation_reasoning", "pipeline_adaption")

        graph.set_entry_point("match_schemas")
        self.graph = graph.compile()

        # make tools available to the model based on model type
        self.tools = {profileTool.name: profileTool}
#        if USE_LLM == "groq" or USE_LLM == "gpt" or USE_LLM == "gemini":
        self.model = model.bind_tools([profileTool])
#        elif USE_LLM == "gemini_broken":
#            self.model = model.bind_tools(profileTool)
        

    def _invoke_with_usage(self, message, tag):
        with get_openai_callback() as cb:
            result = self.model.invoke(message)
        self.token_usage["prompt_tokens"] += cb.prompt_tokens
        self.token_usage["completion_tokens"] += cb.completion_tokens
        self.token_usage["total_tokens"] += cb.total_tokens
        self.token_usage["total_cost"] += cb.total_cost
        print(f"TOKEN USAGE ({tag}):")
        print(f"   Prompt tokens: {cb.prompt_tokens:,}")
        print(f"   Completion tokens: {cb.completion_tokens:,}")
        print(f"   Total tokens: {cb.total_tokens:,}")
        print(f"   Estimated cost: ${cb.total_cost:.6f}")
        return result

    def _print_total_usage(self):
        t = self.token_usage
        print("TOTAL TOKEN USAGE:")
        print(f"   Prompt tokens: {t['prompt_tokens']:,}")
        print(f"   Completion tokens: {t['completion_tokens']:,}")
        print(f"   Total tokens: {t['total_tokens']:,}")
        print(f"   Estimated cost: ${t['total_cost']:.6f}")

    # Creates tool calls to profile the data and saves it into agent state 
    def match_schemas(self, state: SimpleModelAgentState):
        self.logger.info("----------------------- Entering match_schemas -----------------------")
        if state.get("schema_correspondences"):
            print("[*] Schema correspondences already present; skipping match_schemas")
            return {}

        print("[*] Running schema matching")
        result = run_schema_matching(
            dataset_paths=state["datasets"],
            model=self.base_model,
            output_dir="output/schema-matching",
        )
        self.logger.info("Leaving match_schemas")
        return result

    def profile_data(self, state:SimpleModelAgentState):
        self.logger.info('----------------------- Entering profile_data -----------------------')

        print("[*] Profiling datasets")

        system_prompt = """
            You are a data scientist tasked with the integration of several datasets.
            For each dataset path provided, call the tool `profile_dataset` with the path
            (one tool call per dataset).
        """
        
        datasets_list_str = "\n".join(state['datasets'])
        human_content = f"Please profile these datasets (one call per dataset):\n{datasets_list_str}"
        message = [SystemMessage(content=system_prompt), HumanMessage(content=human_content)]
        self.logger.info("Input Message:" + str(message))
        
        result = self._invoke_with_usage(message, "profile_data")
        self.logger.info("RESULT:" + str(result))

        # call tools
        tool_calls = result.tool_calls

        self.logger.info("Tool Calls:" + str(tool_calls))
        results = {}
        for t in tool_calls:
            if not t['name'] in self.tools:      # check for bad tool name from LLM
                self.logger.info("adapt_pipeline: ....bad tool name....")
                result = "bad tool name, retry"  # instruct LLM to retry if bad
            else:
                result = self.tools[t['name']].invoke(t['args'])
            
#            if USE_LLM == "groq" or USE_LLM == "gpt" or USE_LLM == "gemini":
            results[t['args']['path']] = result
#            elif USE_LLM == "gemini_broken":
#                results[t['args']['__arg1']] = result

        with open(OUTPUT_DIR + "profile/profiles.json", "w") as file:
            file.write(json.dumps(results, indent=2))
          
        self.logger.info('Leaving profile_data')
        return {'data_profiles': results}

    def run_blocking_tester(self, state: SimpleModelAgentState):
        self.logger.info("----------------------- Entering run_blocking_tester -----------------------")

        if SKIP_BLOCKING_TESTER:
            path = Path(OUTPUT_DIR + "blocking-evaluation/blocking_config.json")
            if path.exists():
                with path.open("r", encoding="utf-8") as f:
                    blocking_config = json.load(f)
                    print("[+] Using saved blocking strategy: " + json.dumps(blocking_config, indent=2))
                    return {"blocking_config": blocking_config}
            else:
                print("[-] Cannot skip blocking tester. No saved blocking strategy found")
        
        if state.get("blocking_config"):
            print("[*] Blocking config already present in state; skipping BlockingTester")
            return {}

        print("[*] Running BlockingTester")
        tester = BlockingTester(
            llm=self.base_model,
            datasets=state["datasets"],
            max_candidates=350000,
            blocking_testsets=state['entity_matching_testsets'],
            output_dir="output/blocking-evaluation",
            pc_threshold=0.9,
            max_attempts=5,
            max_error_retries=2,
            verbose=True
        )
        _, blocking_config = tester.run_all_pairs()
        print("[*] Blocking config computed:\n" + json.dumps(blocking_config, indent=2))
        self.logger.info("Leaving run_blocking_tester")
        return {"blocking_config": blocking_config}

    def run_matching_tester(self, state: SimpleModelAgentState):
        self.logger.info("----------------------- Entering run_matching_tester -----------------------")

        if SKIP_MATCHING_TESTER:
            path = Path(OUTPUT_DIR + "matching-evaluation/matching_config.json")
            if path.exists():
                with path.open("r", encoding="utf-8") as f:
                    matching_config = json.load(f)
                    print("[+] Using saved matching strategy: " + json.dumps(matching_config, indent=2))
                    return {"matching_config": matching_config}
            else:
                print("[-] Cannot skip matching tester. No saved matching strategy found")
        
        if state.get("matching_config"):
            print("[*] Matching config already present in state; skipping MatchingTester")
            return {}

        print("[*] Running MatchingTester")
        tester = MatchingTester(
            llm=self.base_model,
            datasets=state["datasets"],
            matching_testsets=state['entity_matching_testsets'],
            blocking_config=state.get("blocking_config", {}),
            output_dir="output/matching-evaluation",
            f1_threshold=0.75,
            max_attempts=4,
            max_error_retries=2,
            verbose=True,
            matcher_mode=state.get("matcher_mode", "ml"),
        )
        _, matching_config = tester.run_all()
        print("[*] Matching config computed:\n" + json.dumps(matching_config, indent=2))
        self.logger.info("Leaving run_matching_tester")
        return {"matching_config": matching_config}


    def pipeline_adaption(self, state: SimpleModelAgentState):
        self.logger.info('----------------------- Entering pipeline_adaption -----------------------')

        ####### PREPARE SYSTEM PROMT #######
    
        # Load example pipeline code
        matcher_mode = state.get("matcher_mode", "ml")
        example_name = "example_pipeline_ml.py" if matcher_mode == "ml" else "example_pipeline.py"
        example_pipeline_path = os.path.join(INPUT_DIR, f"example_pipelines/{example_name}")
        if not os.path.exists(example_pipeline_path):
            raise FileNotFoundError(f"Example pipeline not found at: {example_pipeline_path}")
    
        with open(example_pipeline_path, "r", encoding="utf-8") as f:
            example_pipeline_code = f.read()

        # Prepare first-row previews for each dataset
        dataset_previews = {}
        for path in state['datasets']:
            df = load_dataset(path)
            if df is not None and not df.empty:
                # Take only first row and convert to dictionary
                dataset_previews[path] = df.iloc[0].to_dict()
            else:
                dataset_previews[path] = "Failed to load or empty dataset"
                
        # Prepare prompt for the model
        entity_matching_section = ""

        if state["matcher_mode"] == "ml":
            entity_matching_section = f"""
            2b. Entity matching testsets paths:
            {state["entity_matching_testsets"]}
            """
        
        system_prompt = f"""
        You are a data scientist tasked with the integration of several datasets.
        You are provided with the following inputs:

        1. An example integration pipeline (Python code using PyDI library). Pay
        attention to the comments within the pipeline, as they also contain important instructions:
        {example_pipeline_code}
    
        2. A list of dataset file paths:
        {json.dumps(state['datasets'], indent=2)}

        {entity_matching_section}
    
        3. The first row of each dataset to help understand the structure:
        {json.dumps(dataset_previews, indent=2)}
    
        4. A dictionary containing the profile data of the datasets
        (including number of rows, nulls_per_column and dtypes of 
        the columns):
        {json.dumps(state['data_profiles'], indent=2)}
        """
        
        # Include blocking config if available (from BlockingTester)
        blocking_config = state.get('blocking_config')
        if blocking_config:
            system_prompt += f"""

        5. **BLOCKING CONFIGURATION** (pre-computed optimal blocking strategies):
        This configuration was determined by a blocking evaluation agent. Use these settings
        for your blocking step in the pipeline:
        {json.dumps(blocking_config, indent=2)}
        
        IMPORTANT: Use the id_columns and blocking_strategies from this config:
        - Use the correct id_column for each dataset as specified
        - Use the recommended strategy (exact_match_single/multi or semantic_similarity)
        - Use the recommended columns for blocking
        - Strategy to blocker mapping:
          * exact_match_single / exact_match_multi -> StandardBlocker (exact match on columns)
          * token_blocking / ngram_blocking -> TokenBlocker (token or n-gram blocking)
          * sorted_neighbourhood -> SortedNeighbourhoodBlocker (window)
          * semantic_similarity -> EmbeddingBlocker (top_k)
        """

        # Include matching config if available (from MatchingTester)
        matching_config = state.get('matching_config')
        if matching_config:
            if matcher_mode == "ml":
                system_prompt += f"""

        6. **MATCHING CONFIGURATION** (pre-computed comparator settings):
        This configuration was determined by a matching evaluation agent. Use these settings
        for your MLBasedMatcher feature extraction and training:
        {json.dumps(matching_config, indent=2)}

        IMPORTANT: Use the matching_strategies from this config:
        - Build comparators (StringComparator/NumericComparator/DateComparator) for each dataset pair
        - Use the comparators as features in FeatureExtractor
        - Train a classifier on the labeled pairs (labels in the testset) and use MLBasedMatcher
        - Do NOT set weights; ML model learns weights internally
        - Do NOT add RuleBasedMatcher fallback branches
        - Follow the example ML pipeline structure and naming closely; do not invent new variable roles
        - preprocess mapping: "lower" -> str.lower, "strip" -> str.strip, "lower_strip" -> lambda x: str(x).lower().strip()
        """
            else:
                system_prompt += f"""

        6. **MATCHING CONFIGURATION** (pre-computed comparator settings):
        This configuration was determined by a matching evaluation agent. Use these settings
        for your RuleBasedMatcher step in the pipeline:
        {json.dumps(matching_config, indent=2)}

        IMPORTANT: Use the matching_strategies from this config:
        - Build comparators (StringComparator/NumericComparator/DateComparator) for each dataset pair
        - Use the specified weights and threshold
        - preprocess mapping: "lower" -> str.lower, "strip" -> str.strip, "lower_strip" -> lambda x: str(x).lower().strip()
        """

        system_prompt += """

        Your task is to **create a similar integration pipeline** so that it works with
        the datasets provided. Output should only consist of the relevant Python code
        for the integration pipeline.
        """

        evaluation_analysis = state.get("evaluation_analysis", None)
        if evaluation_analysis:
            system_prompt += f"""
            8. Evaluation reasoning from prior pipeline run:
            {evaluation_analysis}
            """
        # Include documentation if global variable is set
        if INCLUDE_DOCS:
            doc_path = os.path.join(INPUT_DIR, "documentation")
            if os.path.exists(doc_path):
                doc_files = [f for f in os.listdir(doc_path) if f.endswith(".md")]
                all_docs = ""
                for f_name in doc_files:
                    with open(os.path.join(doc_path, f_name), "r", encoding="utf-8", errors="replace") as f:
                        all_docs += f.read() + "\n\n"
                system_prompt += "\n\n### Documentation\n" + all_docs

        ####### PREPARE TASK PROMT #######

        # Determine if this is initial generation, fix due to execution error, or fix due to poor evaluation
        if "pipeline_execution_result" not in state and "evaluation_metrics" not in state:
            print("[*] Creating initial pipeline")
            human_content = "Create the integration pipeline for the provided datasets."
    
        else:
            # Either a pipeline execution error or evaluation-based adaption
            broken_pipeline_path = OUTPUT_DIR + "code/pipeline.py"
            with open(broken_pipeline_path, "r", encoding="utf-8", errors="ignore") as f:
                broken_code = f.read()
    
            human_content = "You previously generated Python integration pipeline code.\n"
            
            # Add execution error if exists
            if "pipeline_execution_result" in state and state["pipeline_execution_result"].lower().startswith("error"):
                print("[*] Trying to fix pipeline")
                human_content += f"Executing this pipeline caused the following error:\n{state['pipeline_execution_result']}\n"
    
            # Add evaluation feedback if available
            if "evaluation_metrics" in state:
                print("[*] Trying to improve pipeline based on evaluation")
                human_content += f"Evaluation of the last run shows the following metrics:\n{json.dumps(state['evaluation_metrics'], indent=2)}\n"
                human_content += "Improve the pipeline to increase overall accuracy and correct errors highlighted by the evaluation.\n"
    
            human_content += "Here is the current pipeline code:\n" + broken_code
            human_content += "\nOutput ONLY the corrected Python code."
            
        ####### EXECUTE PROMT #######
        
        message = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=human_content)
        ]
    
        self.logger.info("Input Message:" + str(message))
    
        # Call the model
        adapted_pipeline = self._invoke_with_usage(message, "pipeline_adaption")
        adapted_pipeline = adapted_pipeline.content if hasattr(adapted_pipeline, "content") else str(adapted_pipeline)

        self.logger.info("RESULT:" + str(adapted_pipeline))
        
        # Strip leading/trailing ``` if present
        if adapted_pipeline.startswith("```python") and adapted_pipeline.endswith("```"):
            adapted_pipeline = adapted_pipeline[9:-3].strip()
        
        with open(OUTPUT_DIR + "code/pipeline.py", 'w', errors="ignore") as file:
            file.write(str(adapted_pipeline))
    
        self.logger.info('Leaving pipeline_adaption')
        return {'integration_pipeline_code': adapted_pipeline}

    def execute_pipeline(self, state: SimpleModelAgentState):
        self.logger.info('----------------------- Entering execute_pipeline -----------------------')

        print("[*] Running generated pipeline")

        attempts = state.get("pipeline_execution_attempts", 0) + 1
        state["pipeline_execution_attempts"] = attempts
    
        pipeline_path = OUTPUT_DIR + "code/pipeline.py"
    
        try:
            result = subprocess.run(
                [sys.executable, pipeline_path],
                capture_output=True,
                text=True,
                timeout=3600
            )
    
            if result.returncode == 0:
                print("[+] Pipeline successfully completed")
                execution_result = "success"
            else:
                execution_result = f"error: {result.stderr}"[:10000]
                print("Error: " + str(execution_result))
    
        except Exception as e:
            execution_result = f"error: {str(e)}"
    
        self.logger.info("Pipeline execution result: " + execution_result)
        self.logger.info('Leaving execute_pipeline')
    
        return {
            "pipeline_execution_result": execution_result,
            "pipeline_execution_attempts": attempts
        }
        
    def evaluation_adaption(self, state: SimpleModelAgentState):
        self.logger.info('----------------------- Entering evaluation_adaption -----------------------')
        
        example_eval_path = INPUT_DIR + "example_pipelines/example_evaluation.py"
        example_eval_code = open(example_eval_path).read()
    
        fused_output_path = "output/data_fusion/fusion_data.csv"

        # Prepare first-row preview for testset
        dataset = load_dataset(state['fusion_testset'])
        if dataset is not None and not dataset.empty:
            # Take only first row and convert to dictionary
            testset_preview = dataset.iloc[0].to_dict()
        else:
            testset_preview = "Failed to load or empty dataset"
    
        system_prompt = f"""
        You are a data scientist evaluating a data fusion pipeline.
    
        Example evaluation code:
        {example_eval_code}
    
        Generated integration pipeline:
        {state['integration_pipeline_code']}
    
        Dataset profiles:
        {json.dumps(state['data_profiles'], indent=2)}
    
        The fused output is located at:
        {fused_output_path}

        The testset is located at:
        {state['fusion_testset']}

        The first row of the testset looks like the following:
        {testset_preview}
    
        Create evaluation code that:
        - Uses the correct fusion strategy
        - Loads the fused output
        - Loads the gold standard test set
        - Prints structured evaluation metrics
        - Prints chosen evaluation functions in a compact one-line summary
        """

        evaluation_analysis = state.get("evaluation_analysis", None)
        if evaluation_analysis:
            system_prompt += f"""
            Evaluation reasoning from prior pipeline run:
            {evaluation_analysis}
            """

        ####### HUMAN PROMPT #######
        if not state.get("evaluation_execution_result"):
            print("[*] Adapting evaluation code")
            # First generation
            human_content = """
            Create the evaluation code.
            """
        else:
            # Fix broken evaluation
            print("[*] Fixing evaluation code")
            attempts = state.get("evaluation_execution_attempts", 0)
    
            if attempts >= 3:
                self.logger.info("Maximum evaluation fix attempts reached")
                return {"evaluation_execution_result": "error: max attempts reached"}
    
            eval_path = OUTPUT_DIR + "code/evaluation.py"
            with open(eval_path, "r", encoding="utf-8") as f:
                broken_code = f.read()
    
            error = state["evaluation_execution_result"]
    
            human_content = f"""
            You previously generated Python evaluation code.
            Executing this code caused the following error:
    
            {error}
    
            Here is the current evaluation code:
            {broken_code}
    
            Fix the code so that it executes successfully.
            Output ONLY the corrected Python code.
            """
    
        ####### MODEL CALL #######
    
        message = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=human_content)
        ]
    
        self.logger.info("Input Message:" + str(message))
    
        result = self._invoke_with_usage(message, "evaluation_adaption")
        code = result.content.strip("```python").strip("```")
    
        self.logger.info("RESULT:" + str(code))
    
        with open(OUTPUT_DIR + "code/evaluation.py", "w") as f:
            f.write(code)

        self.logger.info('Leaving evaluation_adaption')
    
        return {"evaluation_code": code}

    def execute_evaluation(self, state: SimpleModelAgentState):
        self.logger.info('----------------------- Entering execute_evaluation -----------------------')
    
        print("[*] Running generated evaluation")
    
        attempts = state.get("evaluation_execution_attempts", 0) + 1
        state["evaluation_execution_attempts"] = attempts
    
        evaluation_path = OUTPUT_DIR + "code/evaluation.py"
    
        try:
            result = subprocess.run(
                [sys.executable, evaluation_path],
                capture_output=True,
                text=True,
                timeout=3600
            )
    
            if result.returncode == 0:
                print("[+] Evaluation successfully completed")
                execution_result = "success"
            else:
                print("[-] Evaluation could not be executed")
                execution_result = f"error: {result.stderr}"[:10000]
                print("Error: " + str(execution_result))
    
        except Exception as e:
            execution_result = f"error: {str(e)}"
    
        self.logger.info("Evaluation execution result: " + execution_result)
        self.logger.info('Leaving execute_evaluation')
    
        return {
            "evaluation_execution_result": execution_result,
            "evaluation_execution_attempts": attempts
        }

    def evaluation_decision(self, state: SimpleModelAgentState):
        self.logger.info('----------------------- Entering evaluation_decision -----------------------')
        print("[*] Reading evaluation metrics")

        attempts = state.get("evaluation_attempts", 0) + 1
    
        eval_path = "output/pipeline_evaluation/pipeline_evaluation.json"
    
        if not os.path.exists(eval_path):
            self.logger.warning("Evaluation file missing")
            metrics = {"error": "evaluation file missing"}
        else:
            with open(eval_path, "r", encoding="utf-8") as f:
                metrics = json.load(f)
    
        self.logger.info(f"Evaluation metrics: {metrics}")
        print(f"Evaluation metrics: {json.dumps(metrics)}")
    
        # Store metrics in the state
        state["evaluation_metrics"] = metrics

        overall_acc = metrics.get("overall_accuracy", 0.0)
        print(f"[*] Overall Accuracy: {overall_acc:.3%}")

        if overall_acc >= 0.85 or attempts >= 3:
            self._print_total_usage()
    
        self.logger.info('Leaving evaluation_decision')

        return {
            "evaluation_metrics": metrics,
            "evaluation_attempts": attempts,
            "pipeline_execution_result": "",
            "pipeline_execution_attempts": 0,
            "evaluation_execution_result": "",
            "evaluation_execution_attempts": 0
        }
    def evaluation_reasoning(self, state: SimpleModelAgentState):
        self.logger.info('----------------------- Entering evaluation_reasoning -----------------------')
    
        debug_path = "output/pipeline_evaluation/debug_fusion_eval.jsonl"
        debug_events = []
    
        if os.path.exists(debug_path):
            with open(debug_path, "r", encoding="utf-8") as f:
                for line in f:
                    try:
                        debug_events.append(json.loads(line))
                    except Exception:
                        continue
    
        # Prepare system prompt
        system_prompt = """
        You are a data integration expert analyzing the evaluation results of a data integration pipeline.
    
        You are given:
        - Aggregate evaluation metrics
        - Detailed per-record debugging events from a fusion evaluation
        - The current integration pipeline code
        - The optimal blocking and matching configurations
    
        Your task:
        1. Identify the dominant error types (false positives, false negatives, missed blocks, bad matches)
        2. Determine whether errors are more likely caused by:
           - Blocking being too restrictive or too loose
           - Matching thresholds or comparator choices
           - Schema or preprocessing issues
           - Logic in the current pipeline code
        3. Suggest concrete, actionable improvements for the next pipeline iteration

        IMPORTANT: The blocking and matching strategies have been determined earlier and have been already evaluated. Therefore, 
        focus on improving the fusion of the pipeline.
    
        Respond with concise, structured reasoning.
        """
    
        # Prepare human content
        human_content = f"""
        Evaluation metrics:
        {json.dumps(state.get("evaluation_metrics", {}), indent=2)}
    
        Debug fusion events (JSONL):
        {json.dumps(debug_events[:50], indent=2)}
    
        Current integration pipeline code:
        {state.get("integration_pipeline_code", "Pipeline code not available")}

        Current pipeline evaluation code:
        {state.get("evaluation_code", "Pipeline evaluation code not available")}
    
        Optimal blocking configuration:
        {json.dumps(state.get("blocking_config", {}), indent=2)}
    
        Optimal matching configuration:
        {json.dumps(state.get("matching_config", {}), indent=2)}
        """
    
        # Model call
        message = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=human_content)
        ]
    
        result = self.model.invoke(message)
        analysis = result.content if hasattr(result, "content") else str(result)
    
        self.logger.info("Evaluation reasoning produced")
        print("[*] Evaluation reasoning:\n" + analysis)
    
        return {"evaluation_analysis": analysis}
    


## Invoke Pipeline

In [92]:
if USE_LLM == "gemini":
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash",
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
    )
elif USE_LLM == "groq":
    llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0.2
    )
elif USE_LLM == "gpt":
    llm = ChatOpenAI(
        model="gpt-5.1",
        temperature=0,
        max_tokens=None,
    )

In [53]:
# Music use case
entity_matching_testsets = {
    ("discogs", "lastfm"): INPUT_DIR + "datasets/music/testsets/discogs_lastfm_goldstandard_blocking.csv",
    ("discogs", "musicbrainz"): INPUT_DIR + "datasets/music/testsets/discogs_musicbrainz_goldstandard_blocking.csv",
    ("musicbrainz", "lastfm"): INPUT_DIR + "datasets/music/testsets/musicbrainz_lastfm_goldstandard_blocking.csv",
}
datasets = [
    INPUT_DIR + "datasets/music/discogs.xml",
    INPUT_DIR + "datasets/music/lastfm.xml",
    INPUT_DIR + "datasets/music/musicbrainz.xml"
]
fusion_testset = INPUT_DIR + "datasets/music/testsets/test_set.xml"

In [41]:
# Restaurants use case
entity_matching_testsets = {
    ("kaggle_small", "uber_eats_small"): INPUT_DIR + "datasets/restaurant/testsets/kaggle_uber_eats_goldstandard_blocking_small.csv",
    ("kaggle_small", "yelp_small"): INPUT_DIR + "datasets/restaurant/testsets/kaggle_yelp_goldstandard_blocking_small.csv",
    ("uber_eats_small", "yelp_small"): INPUT_DIR + "datasets/restaurant/testsets/uber_eats_yelp_goldstandard_blocking_small.csv",
}

datasets = [
    INPUT_DIR + "datasets/restaurant/kaggle_small.parquet",
    INPUT_DIR + "datasets/restaurant/uber_eats_small.parquet",
    INPUT_DIR + "datasets/restaurant/yelp_small.parquet"
]
fusion_testset = INPUT_DIR + "datasets/restaurant/testsets/Restaurant_Fusion_Test_Set.csv"

In [25]:
# Games use case
entity_matching_testsets= {
    ("dbpedia","sales"): INPUT_DIR + "datasets/games/testsets/dpedia_games_sales_goldstandard_blocking.csv",
    ("dbpedia","metacritic"): INPUT_DIR + "datasets/games/testsets/dpedia_games_metacritic_goldstandard_blocking.csv",
    ("sales","metacritic"): INPUT_DIR + "datasets/games/testsets/sales_metacritic_goldstandard_blocking.csv",
}

datasets= [
    INPUT_DIR + "datasets/games/dbpedia.xml",
    INPUT_DIR + "datasets/games/metacritic.xml",
    INPUT_DIR + "datasets/games/sales.xml"
]
fusion_testset= INPUT_DIR + "datasets/games/testsets/test_set.xml"

In [93]:
# Books use case
entity_matching_testsets= {
    ("goodreads_small","amazon_small"): INPUT_DIR + "datasets/books/testsets/goodreads_2_amazon.csv",
    ("metabooks_small","amazon_small"): INPUT_DIR + "datasets/books/testsets/metabooks_2_amazon.csv",
    ("metabooks_small","goodreads_small"): INPUT_DIR + "datasets/books/testsets/metabooks_2_goodreads.csv",
}

datasets= [
    INPUT_DIR + "datasets/books/amazon_small.parquet",
    INPUT_DIR + "datasets/books/goodreads_small.parquet",
    INPUT_DIR + "datasets/books/metabooks_small.parquet"
]
fusion_testset= INPUT_DIR + "datasets/books/testsets/golden_fused_books.csv"

In [94]:
# Skip Blocking and Matching node and use existing strategies saved in 
# "output/blocking-evaluation/blocking_config.json" and "output/matching-evaluation/matching_config.json"
SKIP_BLOCKING_TESTER=True
SKIP_MATCHING_TESTER=True

In [95]:
# prepare agent
agent = SimpleModelAgent(llm, ProfileDatasetTool())

# invoke agent
result = agent.graph.invoke({
    "datasets": datasets,
    "entity_matching_testsets": entity_matching_testsets,
    "fusion_testset": fusion_testset,
    "matcher_mode": "ml"
},
config={"recursion_limit": 200})

[*] Running schema matching
[*] Schema alignment columns:
    amazon_small: ['id', 'title', 'author', 'publish_year', 'publisher', 'isbn_clean']
    goodreads_small: ['id', 'title', 'author', 'rating', 'numratings', 'language', 'genres', 'bookformat', 'edition', 'page_count', 'publisher', 'publish_year', 'price', 'isbn_clean']
    metabooks_small: ['id', 'title', 'author', 'rating', 'numratings', 'language', 'genres', 'publisher', 'page_count', 'price', 'publish_year', 'isbn_clean']
[*] Profiling datasets
TOKEN USAGE (profile_data):
   Prompt tokens: 241
   Completion tokens: 94
   Total tokens: 335
   Estimated cost: $0.000000
[+] Using saved blocking strategy: {
  "blocking_strategies": {
    "goodreads_small_amazon_small": {
      "strategy": "exact_match_single",
      "columns": [
        "isbn_clean"
      ],
      "params": {},
      "pair_completeness": 1.0,
      "num_candidates": 576,
      "is_acceptable": true
    },
    "metabooks_small_amazon_small": {
      "strategy": "